# Data Preperation: California Housing Prices

In [ ]:
import os

import pandas as pd
import numpy as np

import matplotlib.pyplot as plt

from sklearn.impute import SimpleImputer
from sklearn.model_selection import train_test_split
from sklearn.feature_selection import mutual_info_regression
from sklearn.preprocessing import StandardScaler

In [ ]:
path = os.path.join(os.getcwd(), "..", "..", "datasets", "housing.csv")
df = pd.read_csv(path, sep=",")

### Split Train/test

In [ ]:
X = df.drop("median_house_value", axis=1)
y = df["median_house_value"]

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42
)

### Handle Missing Values

In [ ]:
df.isna().sum()

In [ ]:
# impute missing values using simple imputer
imputer = SimpleImputer(strategy="mean")
X_train["total_bedrooms"] = imputer.fit_transform(X_train[["total_bedrooms"]])
X_test["total_bedrooms"] = imputer.transform(X_test[["total_bedrooms"]])

## Categorical Features

`ocean_proximity`

#### remove the value ISLAND

In [ ]:
X_train = X_train[X_train["ocean_proximity"] != "ISLAND"]
X_train["ocean_proximity"].value_counts()

In [ ]:
X_test = X_test[X_test["ocean_proximity"] != "ISLAND"]

#### one hot endoing

In [ ]:
X_train = pd.get_dummies(X_train, columns=["ocean_proximity"], drop_first=False)
X_test = pd.get_dummies(X_test, columns=["ocean_proximity"], drop_first=False)

## Numerical Features

#### handle skeweness & outliers

In [ ]:
num_features = [
    "total_bedrooms",
    "population",
    "households",
    "total_rooms",
    "median_income",
]

In [ ]:
for col in num_features:
    X_train[col] = np.log1p(X_train[col])
    X_test[col] = np.log1p(X_test[col])

In [ ]:
n = len(num_features)
ncols = 4
nrows = -(-n // ncols)  # ceiling division

fig, axes = plt.subplots(nrows, ncols, figsize=(18, nrows * 4))
axes = axes.flatten()

for i, col in enumerate(num_features):
    axes[i].boxplot(X_train[col])
    axes[i].set_title(col)

plt.suptitle("Boxplots of Numerical Features", fontsize=14, y=1.01)
plt.tight_layout()
plt.show()

In [ ]:
# Count outliers for each numerical feature using the IQR rule
outlier_counts = {}

for col in num_features:
    q1 = X_train[col].quantile(0.25)
    q3 = X_train[col].quantile(0.75)
    iqr = q3 - q1
    lower_bound = q1 - 1.5 * iqr
    upper_bound = q3 + 1.5 * iqr
    outlier_mask = (X_train[col] < lower_bound) | (X_train[col] > upper_bound)
    outlier_counts[col] = outlier_mask.sum()

outlier_counts

In [ ]:
outlier_counts = {}

for col in num_features:
    q1 = X_test[col].quantile(0.25)
    q3 = X_test[col].quantile(0.75)
    iqr = q3 - q1
    lower_bound = q1 - 1.5 * iqr
    upper_bound = q3 + 1.5 * iqr
    outlier_mask = (X_test[col] < lower_bound) | (X_test[col] > upper_bound)
    outlier_counts[col] = outlier_mask.sum()

outlier_counts

In [ ]:
# Remove outliers from X_train/X_test using train-based IQR bounds
bounds = {}
for col in num_features:
    q1 = X_train[col].quantile(0.25)
    q3 = X_train[col].quantile(0.75)
    iqr = q3 - q1
    lower = q1 - 1.5 * iqr
    upper = q3 + 1.5 * iqr
    bounds[col] = (lower, upper)

X_train, y_train = X_train.align(y_train, axis=0)
X_test, y_test = X_test.align(y_test, axis=0)

mask_train = pd.Series(True, index=X_train.index)
mask_test = pd.Series(True, index=X_test.index)
for col, (lower, upper) in bounds.items():
    mask_train &= X_train[col].between(lower, upper)
    mask_test &= X_test[col].between(lower, upper)

removed_train = len(X_train) - mask_train.sum()
removed_test = len(X_test) - mask_test.sum()

X_train = X_train[mask_train].copy()
y_train = y_train[mask_train].copy()
X_test = X_test[mask_test].copy()
y_test = y_test[mask_test].copy()

{"removed_train": removed_train, "removed_test": removed_test}

In [ ]:
n = len(num_features)
ncols = 4
nrows = -(-n // ncols)  # ceiling division

fig, axes = plt.subplots(nrows, ncols, figsize=(18, nrows * 4))
axes = axes.flatten()

for i, col in enumerate(num_features):
    axes[i].boxplot(X_train[col])
    axes[i].set_title(col)

plt.suptitle("Boxplots of Numerical Features", fontsize=14, y=1.01)
plt.tight_layout()
plt.show()

In [ ]:
n = len(num_features)
ncols = 4
nrows = -(-n // ncols)  # ceiling division

fig, axes = plt.subplots(nrows, ncols, figsize=(18, nrows * 4))
axes = axes.flatten()

for i, col in enumerate(num_features):
    axes[i].boxplot(X_test[col])
    axes[i].set_title(col)

plt.suptitle("Boxplots of X_test Numerical Features", fontsize=14, y=1.01)
plt.tight_layout()
plt.show()

#### add the new features

In [ ]:
X_train["rooms_per_household"] = X_train["total_rooms"] / X_train["households"]
X_train["bedrooms_per_household"] = X_train["total_bedrooms"] / X_train["households"]
X_train["rooms_per_person"] = X_train["total_rooms"] / X_train["population"]
X_train["bedrooms_per_person"] = X_train["total_bedrooms"] / X_train["population"]
X_train["bedrooms_fraction"] = X_train["total_bedrooms"] / X_train["total_rooms"]
X_train["people_per_household"] = X_train["population"] / X_train["households"]

X_test["rooms_per_household"] = X_test["total_rooms"] / X_test["households"]
X_test["bedrooms_per_household"] = X_test["total_bedrooms"] / X_test["households"]
X_test["rooms_per_person"] = X_test["total_rooms"] / X_test["population"]
X_test["bedrooms_per_person"] = X_test["total_bedrooms"] / X_test["population"]
X_test["bedrooms_fraction"] = X_test["total_bedrooms"] / X_test["total_rooms"]
X_test["people_per_household"] = X_test["population"] / X_test["households"]

### Feature Selection

In [ ]:
train_corr = pd.concat([X_train, y_train], axis=1)
feature_corr = (
    train_corr.corr()["median_house_value"]
    .drop("median_house_value")
    .abs()
    .sort_values(ascending=False)
)
print(feature_corr)

fig, ax = plt.subplots(figsize=(10, max(6, len(feature_corr) * 0.25)))
feature_corr.plot(kind="barh", ax=ax, color="steelblue")
ax.set_title("Absolute Correlation with Median House Value")
ax.set_xlabel("Absolute Pearson Correlation")
ax.invert_yaxis()
plt.tight_layout()
plt.show()

In [ ]:
# Feature selection using mutual information
mi_scores = mutual_info_regression(X_train, y_train, random_state=42)
mi_scores = pd.Series(mi_scores, index=X_train.columns).sort_values(ascending=False)
print(mi_scores)

fig, ax = plt.subplots(figsize=(10, max(6, len(mi_scores) * 0.25)))
mi_scores.plot(kind="barh", ax=ax, color="steelblue")
ax.set_title("Mutual Information Scores with Median House Value")
ax.set_xlabel("Mutual Information")
ax.invert_yaxis()
plt.tight_layout()
plt.show()

-> keep all features

### Scaling features

In [ ]:
# Scale numeric features using StandardScaler
numeric_cols = [
    col for col in X_train.columns if not col.startswith("ocean_proximity_")
]

scaler = StandardScaler()
X_train[numeric_cols] = scaler.fit_transform(X_train[numeric_cols])
X_test[numeric_cols] = scaler.transform(X_test[numeric_cols])

# Scale y_train and y_test
scaler_y = StandardScaler()
y_train = scaler_y.fit_transform(y_train.values.reshape(-1, 1)).flatten()
y_test = scaler_y.transform(y_test.values.reshape(-1, 1)).flatten()

In [ ]:
# Convert boolean columns to int
bool_cols = X_train.select_dtypes(include="object").columns
X_train[bool_cols] = X_train[bool_cols].astype(int)
X_test[bool_cols] = X_test[bool_cols].astype(int)

In [ ]:
X_train.isna().sum()

In [ ]:
from datasets import Dataset

# Convert y to DataFrame
y_train_df = pd.DataFrame(y_train, columns=["median_house_value"])
y_test_df = pd.DataFrame(y_test, columns=["median_house_value"])

# Combine X and y
train_df = pd.concat([X_train.reset_index(drop=True), y_train_df], axis=1)
test_df = pd.concat([X_test.reset_index(drop=True), y_test_df], axis=1).reset_index(
    drop=True
)
test_df.isna().sum()
# Convert to HF Dataset
# train_dataset = Dataset.from_pandas(train_df,preserve_index=False)
# test_dataset  = Dataset.from_pandas(test_df,preserve_index=False)

### Upload to HF

In [ ]:
def save_to_hf(X_train, X_test, y_train, y_test, repo_name):

    # Convert y to DataFrame
    y_train_df = pd.DataFrame(y_train, columns=["median_house_value"])
    y_test_df = pd.DataFrame(y_test, columns=["median_house_value"])

    # Combine X and y
    train_df = pd.concat([X_train.reset_index(drop=True), y_train_df], axis=1)
    test_df = pd.concat([X_test.reset_index(drop=True), y_test_df], axis=1)

    # Convert to HF Dataset
    train_dataset = Dataset.from_pandas(train_df, preserve_index=False)
    test_dataset = Dataset.from_pandas(test_df, preserve_index=False)

    # Push to Hugging Face Hub
    train_dataset.push_to_hub(repo_name, split="train")
    test_dataset.push_to_hub(repo_name, split="test")

    print("Uploaded dataset to Hugging Face:")
    print(repo_name)
    print("Train shape:", train_df.shape)
    print("Test shape:", test_df.shape)

In [ ]:
from huggingface_hub import login

login()

In [ ]:
save_to_hf(
    X_train, X_test, y_train, y_test, repo_name="narimanee/Housing-prices-federated"
)